In [10]:
# Imports
import pandas as pd
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
# 1. Load Dataset
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

In [5]:
# EDA - Distribution
print("Label Distribution")
print(df['label'].value_counts())
print(f"Total messages: {len(df)}")

Label Distribution
label
ham     4825
spam     747
Name: count, dtype: int64
Total messages: 5572


In [6]:
# 2. Cleaning Pipeline
def clean_text(text):
    # Remove non-ASCII characters and broken encoding
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Detect and obfuscate phone numbers
    # Patterns: 123-456-7890, (123) 456-7890, 1234567890, +1 123 456 7890.
    phone_patterns = [
        r'\+?\d{1,3}[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',
        r'\b\d{10,}\b',
        r'\(\d{3}\)\s*\d{3}[-.\s]?\d{4}',
        r'\d{3}[-.\s]\d{3}[-.\s]\d{4}'
    ]
    for pattern in phone_patterns:
        text = re.sub(pattern, '<PHONE>', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

df['cleaned_message'] = df['message'].apply(clean_text)

In [7]:
# 3. Fuzzy Deduplication using TF-IDF + Cosine Similarity
def deduplicate_fuzzy(df, text_column='cleaned_message', threshold=0.85):
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 3))
    tfidf_matrix = vectorizer.fit_transform(df[text_column])

    cosine_sim = cosine_similarity(tfidf_matrix)

    duplicates = set()
    n = len(df)

    for i in range(n):
        if i in duplicates:
            continue
        for j in range(i + 1, n):
            if j in duplicates:
                continue
            if cosine_sim[i, j] >= threshold:
                duplicates.add(j)

    return df.drop(df.index[list(duplicates)])

original_count = len(df)
df_deduplicated = deduplicate_fuzzy(df)
deduplicated_count = len(df_deduplicated)

In [9]:
# 4. Output
df_deduplicated[['label', 'cleaned_message']].to_csv('cleaned_sms_spam.csv', index=False)

print("Summary")
print(f"Original Row Count: {original_count}")
print(f"Cleaned/Deduplicated Row Count: {deduplicated_count}")
print(f"Removed Duplicates: {original_count - deduplicated_count}")
print(f"Cleaned dataset saved to 'cleaned_sms_spam.csv'")

Summary
Original Row Count: 5572
Cleaned/Deduplicated Row Count: 5009
Removed Duplicates: 563
Cleaned dataset saved to 'cleaned_sms_spam.csv'
